In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")
groq_api_key



'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'

In [5]:
from langchain_groq import ChatGroq
model=ChatGroq(groq_api_key=groq_api_key, model="llama-3.1-8b-instant")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000023ADBD41A10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000023ADBD43310>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [6]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="What is the capital of France?")])

AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 42, 'total_tokens': 50, 'completion_time': 0.006984341, 'completion_tokens_details': None, 'prompt_time': 0.002321424, 'prompt_tokens_details': None, 'queue_time': 0.045049295, 'total_time': 0.009305765}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d4ebc-4d5d-7f81-908c-1dbc45e1e401-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 8, 'total_tokens': 50})

In [ ]:

from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="My name is Meem"),
        AIMessage(content="Hello Meem, how can I help you today?"),
        HumanMessage(content="What is my name?")

    ]
)

AIMessage(content='Your name is Meem.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 50, 'total_tokens': 57, 'completion_time': 0.010558251, 'completion_tokens_details': None, 'prompt_time': 0.041596648, 'prompt_tokens_details': None, 'queue_time': 0.136611484, 'total_time': 0.052154899}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d4ebe-99bb-7463-abc2-24bf711c70f4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 50, 'output_tokens': 7, 'total_tokens': 57})

Message Histroy

In [9]:
!pip install langchain_community

In [11]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [12]:
config={"configurable":{"session_id":"chat1"}}

In [13]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Meem and I am an AI Engineer")],
    config=config
)

In [14]:
response.content

'Nice to meet you, Meem. As an AI Engineer, you must be working on some exciting projects. What areas of AI are you focused on, such as natural language processing (NLP), computer vision, robotics, or perhaps something else?'

In [15]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

AIMessage(content='Your name is Meem.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 113, 'total_tokens': 120, 'completion_time': 0.013266952, 'completion_tokens_details': None, 'prompt_time': 0.006742038, 'prompt_tokens_details': None, 'queue_time': 0.045099692, 'total_time': 0.02000899}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d4f36-0f5d-7f42-96ba-9692c0e37b79-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 113, 'output_tokens': 7, 'total_tokens': 120})

In [16]:
## change the config-->session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

"I don't have any information about your name. I'm a large language model, I don't have the ability to retain personal information or recall previous conversations. Each time you interact with me, it's a new conversation and I don't have any prior knowledge about you. If you'd like to share your name, I can refer to you by that name in our conversation."

In [17]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is John")],
    config=config1
)
response.content

"Hello John. It's nice to finally know your name. I can now address you by name in our conversation. How can I assist you today? Do you have any questions or topics you'd like to discuss?"

In [18]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'Your name is John.'

Prompt Template

In [19]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Amnswer all the question to the nest of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [20]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Meem")]})

AIMessage(content="Nice to meet you, Meem. I'm happy to assist you with any questions or topics you'd like to discuss. How's your day going so far?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 58, 'total_tokens': 92, 'completion_time': 0.040938476, 'completion_tokens_details': None, 'prompt_time': 0.003488385, 'prompt_tokens_details': None, 'queue_time': 0.04597843, 'total_time': 0.044426861}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d4f37-7425-7fb3-bedc-0cc726e3f1f6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 58, 'output_tokens': 34, 'total_tokens': 92})

In [21]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

In [23]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Meem")],
    config=config
)

response

AIMessage(content="Nice to meet you, Meem. It's great to chat with you. How's your day going so far? Is there anything I can help you with or would you like to just have a conversation?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 104, 'total_tokens': 147, 'completion_time': 0.061478466, 'completion_tokens_details': None, 'prompt_time': 0.006811647, 'prompt_tokens_details': None, 'queue_time': 0.047119268, 'total_time': 0.068290113}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d4f38-08ce-7740-9562-f1dbb0f7c2a1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 104, 'output_tokens': 43, 'total_tokens': 147})

In [24]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Your name is Meem. We had just started chatting and you introduced yourself.'

In [25]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [26]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Meem")],"language":"Bangla"})
response.content

'নমস্কার! আমার নাম হলো সাহায্যকারী। আপনার নাম মীম, তাই আপনার সাথে কথা বলতে চাই। কী বিষয়ে সমীক্ষা করতে চান?'

In [27]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [28]:
config = {"configurable": {"session_id": "chat4"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Meem")],"language":"Bangla"},
    config=config
)
repsonse.content

'আপনার নাম মীম, আমি আপনার সহায়তার জন্য উপলব্ধ। কি আপনার কথা আছে?'

In [29]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Urdu"},
    config=config,
)

In [30]:
response.content

'آپ کا نام ميم ہے۔'

Managing Conversational Histories

In [43]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [56]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)



In [55]:
response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

"Unfortunately, I'm a large language model, I don't have personal knowledge of your preferences. You'll have to tell me your favorite ice cream flavor!"

In [49]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'You asked me the math problem "2 + 2".'

In [57]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

In [62]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

"I don't think you've told me your name yet!"

In [61]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

'You asked "2 + 2" which is a simple math problem.'